# Seeded stratified sample of GitSkills skills

**MSR 2027 · SkillGuard · RQ1 — sample for the systematic Cisco experiment**

This notebook draws a reproducible, stratified random sample of skills from the GitSkills SQLite database and saves a
**sample manifest** that `01_cisco_scanner_dataset_ready.ipynb` can scan (`SELECTION_MODE = "manifest"`). It only
reads the database (read-only). It never downloads anything, never writes skill files and never runs a scanner.

**Why stratify?** RQ1 asks how the amount of package context changes what Cisco finds. A simple random sample of the
sample database would be dominated by skills that consist of `SKILL.md` alone (about two thirds), leaving few skills
with scripts. Stratifying by the kind of supporting files guarantees enough skills of each kind.

**What is sampled.** The unit is a *distinct* `SKILL.md`: one **representative** per content hash (`file_sha`,
`dedup_primary = 1`). Identical copies of the same `SKILL.md` in other repositories can therefore never be drawn
twice. The frame is built from the database alone; no keyword search, popularity measure, earlier pilot or
security signal influences which skills are drawn.

**How it is drawn.** Within each stratum the eligible representatives are put in a fixed order (by `file_sha`), a
`random.Random(SEED)` generator gives each one a random number, and the skills with the smallest numbers are
selected. Python guarantees that `Random(seed).random()` returns the same sequence for the same integer seed across
versions, so the same database file and the same seed always give the same sample.

**Equal allocation needs weights.** Each stratum gets the same number of skills although the strata have very
different sizes. Counts and proportions *within* a stratum can be reported directly. Any statement about the whole
population (for example "x % of skills trigger a CRITICAL finding") must weight each sampled skill by
`design_weight = stratum population / stratum sample size`, which is saved for every skill.

Outputs (`results/samples/<sample id>/`): `sample_manifest.csv`, `sample_manifest.json`, `sampling_frame.csv` (every
representative with its stratum or exclusion reason and random number), `sampling_metadata.json` and
`sampling_report.md`. Re-running the notebook never overwrites them: it regenerates the sample and checks that it is
identical to the saved one.

## 1 — Settings

In [ ]:
DATABASE = "sample"                 # "sample" or a path to a full GitSkills SQLite file
SEED = 2027                         # fixed random seed
STRATUM_SIZES = {                   # stratum -> number of skills to draw (equal allocation)
    "scripts": 10,                  # at least one executable script in the folder
    "supporting_no_scripts": 10,    # supporting files, none of them scripts or program source
    "skill_md_only": 10,            # the folder contains nothing but SKILL.md
}
OUTPUT_ROOT = "results/samples"

# "Executable script" = a file an agent can run directly with an interpreter, or an extensionless file starting with "#!".
SCRIPT_EXTENSIONS = {"py", "sh", "bash", "zsh", "fish", "ps1", "psm1", "bat", "cmd", "js", "mjs", "cjs", "ts", "mts",
                     "cts", "rb", "pl", "php", "lua", "r", "exs"}
# Program source that normally needs compiling first. Folders with these but no script are ambiguous (excluded below).
PROGRAM_SOURCE_EXTENSIONS = {"go", "java", "rs", "c", "h", "cc", "cpp", "hpp", "cs", "kt", "kts", "swift", "scala", "ex",
                             "jsx", "tsx", "dart", "m", "vb", "fs", "clj"}

# Earlier hand-picked or keyword-selected skills. Used ONLY to report overlap; they do not influence the draw.
PREVIOUS_SELECTIONS = {
    "pilot canary-tokens": ("26zl/cybersec-toolkit", ".claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md"),
    "pilot x": ("AceDataCloud/Skills", "skills/x/SKILL.md"),
    "pilot loa-credentials": ("0xHoneyJar/loa-dixie", ".claude/skills/managing-credentials/SKILL.md"),
    "keyword batch API_KEY #1": ("0xHoneyJar/loa-constructs", ".claude/skills/bridgebuilder-review/SKILL.md"),
    "keyword batch API_KEY #3": ("1CestlaVie/astrbot-plugin-dev-skill", "SKILL.md"),
}
KNOWN_SAMPLE_SHA256 = "683888c923a51952fb66f83f21351ed5bf1127ddddf380e517592ae27c367658"   # official sample (README)

## 2 — Connect (read-only) and identify the dataset

The SHA-256 of the database file is compared with the published hash of the official GitSkills sample. If they match,
every result built on this sample is a **sample-based methodological experiment**, not a representative study of the
full GitSkills population.

In [ ]:
import hashlib
import json
import platform
import random
import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

cwd = Path.cwd().resolve()
ROOT = next(f for f in (cwd, *cwd.parents) if (f / "notebooks").is_dir() and (f / "data").is_dir())
DB_PATH = (ROOT / "data/sample/agent_skills_sample.db") if DATABASE == "sample" else Path(DATABASE).expanduser()
DB_PATH = (DB_PATH if DB_PATH.is_absolute() else ROOT / DB_PATH).resolve()
conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
conn.execute("PRAGMA query_only = ON")
sql = lambda query, params=(): pd.read_sql_query(query, conn, params=params)

digest = hashlib.sha256()
with open(DB_PATH, "rb") as handle:
    for chunk in iter(lambda: handle.read(8 << 20), b""):
        digest.update(chunk)
DB_SHA256 = digest.hexdigest()
DB_TAG = re.sub(r"[^a-zA-Z0-9_-]", "_", DB_PATH.stem)[:45]
IS_OFFICIAL_SAMPLE = DB_SHA256 == KNOWN_SAMPLE_SHA256
DATASET_LABEL = ("official GitSkills sample database (sample-based methodological experiment; NOT representative of the full GitSkills population)"
                 if IS_OFFICIAL_SAMPLE else "unrecognised database file (check whether this is the full GitSkills release)")
print("Database :", DB_PATH.relative_to(ROOT) if DB_PATH.is_relative_to(ROOT) else DB_PATH)
print("SHA-256  :", DB_SHA256)
print("Dataset  :", DATASET_LABEL)

## 3 — The population: occurrences, distinct contents, representatives

* An **occurrence** is one `SKILL.md` file at one repository path.
* Occurrences with identical bytes share a `file_sha` (Git blob hash). Each `file_sha` has exactly one
  **representative** row (`dedup_primary = 1`), which stores the text and the folder contents. The check below
  confirms there is exactly one representative per hash.

In [ ]:
population = sql("""
    SELECT COUNT(*)                                   AS skill_md_occurrences,
           COUNT(DISTINCT file_sha)                   AS distinct_content_hashes,
           SUM(dedup_primary = 1)                     AS representatives,
           SUM(dedup_primary = 1 AND length(trim(COALESCE(content, ''))) > 0) AS representatives_with_text,
           COUNT(DISTINCT repo_full_name)             AS repositories
    FROM artifacts
""").iloc[0].to_dict()
population = {k: int(v) for k, v in population.items()}
reps_per_hash = sql("SELECT file_sha, SUM(dedup_primary) AS reps FROM artifacts GROUP BY file_sha")
assert (reps_per_hash["reps"] == 1).all(), "Some content hashes do not have exactly one representative"
assert population["representatives"] == population["distinct_content_hashes"]
display(pd.Series(population, name="count").to_frame())

## 4 — How GitSkills records a skill folder

These facts decide which strata can be identified reliably:

* `artifacts.composition_fetched = 1` means the folder listing was collected; `0` means no listing exists.
* `artifacts.composition_truncated = 1` means the listing was cut off. For truncated folders `sibling_count` (the true
  file count) is larger than the number of file rows stored, and many report `sibling_count = 0` although the
  listing is unreliable, so "no files" cannot be trusted there.
* `artifact_siblings` stores one row per **file** (`entry_type = 'file'`) and per **directory** (`'dir'`). Directories
  only appear above files; there are no empty-directory-only folders.
* For files, `content_fetched = 1` with text means the content is stored; `2` means it was skipped (binary, oversized,
  capped folder). A few rows have `content_fetched = 1` but no text and a non-zero size: their text is also missing.
* A folder is **complete** (rebuildable on disk) only if the listing is untruncated and every file's text is stored.
  This is exactly the rule `01_cisco_scanner_dataset_ready.ipynb` uses before running full-folder scans.

In [ ]:
display(sql("""SELECT composition_fetched, composition_truncated, COUNT(*) AS representatives,
                      SUM(sibling_count) AS sibling_count_total
               FROM artifacts WHERE dedup_primary = 1 GROUP BY 1, 2"""))
display(sql("""SELECT s.entry_type, s.content_fetched, (s.content IS NOT NULL) AS text_stored, COUNT(*) AS rows
               FROM artifact_siblings s
               JOIN artifacts a ON a.repo_full_name = s.repo_full_name AND a.path = s.artifact_path AND a.dedup_primary = 1
               GROUP BY 1, 2, 3"""))

## 5 — One row per representative: what its folder contains

For every representative we count, from its stored listing: files, executable scripts (by extension, plus
extensionless files beginning with `#!`), program-source files, and files whose text is missing. GitSkills' own
`has_scripts` flag is compared with our script definition; ours is a superset (it also recognises `.mjs`, `.cjs`,
`.ts` and shebang scripts that `has_scripts` misses).

In [ ]:
reps = sql("""
    SELECT a.repo_full_name AS repo, a.path AS skill_path, a.filename, a.file_sha, a.name, a.location_class,
           a.frontmatter_valid, a.sibling_count, a.has_scripts AS has_scripts_gitskills,
           a.has_references AS has_references_gitskills, a.composition_fetched, a.composition_truncated,
           a.first_commit_at, a.last_commit_at, a.commit_count, a.content,
           (SELECT COUNT(*) FROM artifacts c WHERE c.file_sha = a.file_sha) AS copies_in_dataset,
           r.stars AS repo_stars, r.forks AS repo_forks, r.is_fork AS repo_is_fork, r.language AS repo_language,
           r.license AS repo_license, r.created_at AS repo_created_at, r.pushed_at AS repo_pushed_at
    FROM artifacts a LEFT JOIN repos r ON r.full_name = a.repo_full_name
    WHERE a.dedup_primary = 1
""")
files = sql("""
    SELECT s.repo_full_name AS repo, s.artifact_path AS skill_path, s.entry_name,
           (s.content_fetched = 1 AND s.content IS NOT NULL) AS text_stored,
           substr(s.content, 1, 2) = '#!' AS shebang
    FROM artifact_siblings s
    JOIN artifacts a ON a.repo_full_name = s.repo_full_name AND a.path = s.artifact_path AND a.dedup_primary = 1
    WHERE s.entry_type = 'file'
""")
extension = files["entry_name"].str.extract(r"\.([A-Za-z0-9]+)$")[0].str.lower()
files["is_script"] = extension.isin(SCRIPT_EXTENSIONS) | (extension.isna() & (files["shebang"] == 1))
files["is_program_source"] = extension.isin(PROGRAM_SOURCE_EXTENSIONS)
per_folder = files.groupby(["repo", "skill_path"]).agg(
    n_files_listed=("entry_name", "size"), n_script_files=("is_script", "sum"),
    n_program_source_files=("is_program_source", "sum"), n_files_with_text=("text_stored", "sum")).reset_index()

frame = reps.merge(per_folder, on=["repo", "skill_path"], how="left")
for column in ("n_files_listed", "n_script_files", "n_program_source_files", "n_files_with_text"):
    frame[column] = frame[column].fillna(0).astype(int)
frame["skill_md_sha256"] = frame["content"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest())
frame = frame.drop(columns="content")
frame["listing_captured"] = frame["composition_fetched"] == 1
frame["listing_complete"] = frame["listing_captured"] & (frame["composition_truncated"] == 0)
frame["n_files_without_text"] = frame["n_files_listed"] - frame["n_files_with_text"]
frame["folder_complete"] = frame["listing_complete"] & (frame["n_files_without_text"] == 0)
frame["has_scripts"] = frame["n_script_files"] > 0
frame["has_supporting_files"] = frame["n_files_listed"] > 0

print("Our script flag vs GitSkills has_scripts (representatives with a listing):")
display(pd.crosstab(frame.loc[frame.listing_captured, "has_scripts"], frame.loc[frame.listing_captured, "has_scripts_gitskills"],
                    rownames=["has_scripts (ours)"], colnames=["has_scripts (GitSkills)"], margins=True))

## 6 — Strata and exclusions

Each representative goes through these rules **in order**; the first rule that applies decides. Because every
representative stops at exactly one rule, the strata are mutually exclusive by construction (checked below).

| order | rule | result |
|---|---|---|
| 1 | no folder listing (`composition_fetched != 1`) | excluded: listing not captured |
| 2 | at least one executable script in the listing | stratum **scripts** (a truncated listing is allowed: the scripts are certain) |
| 3 | listing truncated (and no script seen) | excluded: cannot confirm "no scripts" or "only SKILL.md" |
| 4 | program source files but no script | excluded: ambiguous (code that needs compiling) |
| 5 | at least one supporting file | stratum **supporting_no_scripts** |
| 6 | no supporting file | stratum **skill_md_only** |

Folder completeness is **not** an inclusion criterion: a skill whose supporting files are partly missing can still
be sampled. Its `SKILL.md`-only scans run normally and its full-folder scans are reported as
`SKIPPED_INCOMPLETE_FOLDER` by the scanning notebook. The tables report how many sampled folders are incomplete.

In [ ]:
STRATUM_DESCRIPTIONS = {
    "scripts": "at least one executable script among the supporting files",
    "supporting_no_scripts": "supporting files, but no executable script and no program source",
    "skill_md_only": "complete listing with no file besides SKILL.md",
}


def assign(row):
    if not row["listing_captured"]:
        return "excluded: folder listing not captured"
    if row["has_scripts"]:
        return "scripts"
    if not row["listing_complete"]:
        return "excluded: listing truncated, script presence unknown"
    if row["n_program_source_files"] > 0:
        return "excluded: program source but no executable script"
    return "supporting_no_scripts" if row["has_supporting_files"] else "skill_md_only"


frame["stratum"] = frame.apply(assign, axis=1)
assert frame["stratum"].notna().all() and len(frame) == population["representatives"]
assert set(frame["stratum"]) - set(STRATUM_SIZES) == {s for s in frame["stratum"] if s.startswith("excluded")}

counts = frame["stratum"].value_counts()
exclusions = {k: int(v) for k, v in counts.items() if k.startswith("excluded")}
strata = pd.DataFrame({
    "description": pd.Series(STRATUM_DESCRIPTIONS),
    "population_N": counts.reindex(list(STRATUM_SIZES)).astype(int),
    "sample_n": pd.Series(STRATUM_SIZES),
    "complete_folders_in_population": frame[frame["folder_complete"]].groupby("stratum").size().reindex(list(STRATUM_SIZES), fill_value=0),
})
strata["sampling_fraction"] = strata["sample_n"] / strata["population_N"]
strata["design_weight"] = strata["population_N"] / strata["sample_n"]
strata["share_complete_folders"] = strata["complete_folders_in_population"] / strata["population_N"]
assert (strata["population_N"] >= strata["sample_n"]).all(), "A stratum is smaller than its requested sample size"
print("Exclusions:", exclusions)
print("Eligible representatives:", int(strata["population_N"].sum()), "of", population["representatives"])
display(strata)

## 7 — Draw the sample

For each stratum separately: sort its representatives by `file_sha`, create `random.Random(SEED)`, give each
representative the next `random()` number (`random_key`) in that order, and select the `sample_n` representatives with
the smallest keys. Every eligible representative keeps its key in `sampling_frame.csv`, so anyone can verify the draw.

In [ ]:
def skill_id_for(repo, skill_path):
    """Same folder name as notebook 01 uses for exported skills: <folder name>-<12 hex of sha256(repo + path)>."""
    stem = re.sub(r"[^a-zA-Z0-9_-]", "-", Path(skill_path).parent.name)[:35].strip("-") or "skill"
    digest = hashlib.sha256(f"{repo}\n{skill_path}".encode()).hexdigest()[:12]
    return f"{stem}-{digest}"


frame["random_key"] = float("nan")
frame["rank_in_stratum"] = pd.NA
frame["selected"] = False
for stratum, n in STRATUM_SIZES.items():
    members = frame[frame["stratum"] == stratum].sort_values("file_sha")
    rng = random.Random(SEED)
    keys = pd.Series([rng.random() for _ in range(len(members))], index=members.index)
    frame.loc[members.index, "random_key"] = keys
    frame.loc[members.index, "rank_in_stratum"] = keys.rank(method="first").astype(int)
    frame.loc[keys.nsmallest(n).index, "selected"] = True

SAMPLE_ID = f"{DB_TAG}_seed{SEED}_" + "_".join(f"{s}{n}" for s, n in STRATUM_SIZES.items())
METHOD = (f"stratified simple random sampling without replacement, equal allocation; unit = representative SKILL.md "
          f"(one per file_sha); per stratum: frame ordered by file_sha, key = random.Random({SEED}).random(), "
          f"smallest keys selected")
frame["skill_id"] = [skill_id_for(r, p) for r, p in zip(frame["repo"], frame["skill_path"])]

sample = frame[frame["selected"]].copy()
sample = sample.merge(strata[["population_N", "sample_n", "sampling_fraction", "design_weight"]],
                      left_on="stratum", right_index=True)
sample["stratum"] = pd.Categorical(sample["stratum"], categories=list(STRATUM_SIZES), ordered=True)
sample = sample.sort_values(["stratum", "rank_in_stratum"]).reset_index(drop=True)
sample["stratum"] = sample["stratum"].astype(str)
sample.insert(0, "sample_position", range(1, len(sample) + 1))
sample.insert(0, "sample_id", SAMPLE_ID)
sample["stratum_description"] = sample["stratum"].map(STRATUM_DESCRIPTIONS)
sample["random_seed"] = SEED
sample["sampling_method"] = METHOD
sample["is_representative"] = True
sample["database_sha256"] = DB_SHA256

MANIFEST_COLUMNS = [
    "sample_id", "sample_position", "stratum", "stratum_description", "population_N", "sample_n", "sampling_fraction",
    "design_weight", "random_seed", "sampling_method", "random_key", "rank_in_stratum",
    "skill_id", "repo", "skill_path", "filename", "file_sha", "skill_md_sha256", "is_representative", "copies_in_dataset",
    "name", "frontmatter_valid", "location_class",
    "has_scripts", "n_script_files", "has_scripts_gitskills", "has_references_gitskills", "n_program_source_files",
    "has_supporting_files", "n_files_listed", "sibling_count", "n_files_with_text", "n_files_without_text",
    "listing_complete", "composition_truncated", "folder_complete",
    "first_commit_at", "last_commit_at", "commit_count",
    "repo_stars", "repo_forks", "repo_is_fork", "repo_language", "repo_license", "repo_created_at", "repo_pushed_at",
    "database_sha256",
]
manifest = sample[MANIFEST_COLUMNS]

# Overlap with earlier hand-picked or keyword-selected skills (reported, never used to include or exclude).
previous = {(r, p): label for label, (r, p) in PREVIOUS_SELECTIONS.items()}
overlap = [{"previous_selection": previous[(r, p)], "repo": r, "skill_path": p}
           for r, p in zip(manifest["repo"], manifest["skill_path"]) if (r, p) in previous]
print(f"Sample {SAMPLE_ID}: {len(manifest)} skills; overlap with earlier pilots/keyword selections: {len(overlap)}")
display(manifest[["sample_position", "stratum", "repo", "skill_path", "n_files_listed", "n_script_files",
                  "folder_complete", "repo_stars", "random_key"]])
display(manifest.groupby("stratum", sort=False).agg(skills=("skill_id", "size"), complete_folders=("folder_complete", "sum"),
                                                    distinct_repos=("repo", "nunique")))

## 8 — Save the manifest (or confirm the saved one is identical)

The first run writes the files. Every later run with the same database and seed regenerates the sample in memory and
compares it with the saved `sample_manifest.csv`; it stops with an error if they differ and never overwrites.

In [ ]:
def sha256_text(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


OUT_DIR = (ROOT / OUTPUT_ROOT / SAMPLE_ID).resolve()
show = lambda path: path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
csv_text = manifest.to_csv(index=False, lineterminator="\n", float_format="%.17g")

if (OUT_DIR / "sample_manifest.csv").is_file():
    saved = (OUT_DIR / "sample_manifest.csv").read_text(encoding="utf-8")
    if saved != csv_text:
        raise ValueError(f"The regenerated sample differs from the saved manifest in {OUT_DIR}. Nothing was overwritten.")
    print(f"Reproducibility check passed: the regenerated sample is byte-identical to {show(OUT_DIR)}/sample_manifest.csv")
else:
    OUT_DIR.mkdir(parents=True, exist_ok=False)
    (OUT_DIR / "sample_manifest.csv").write_text(csv_text, encoding="utf-8")
    frame_columns = ["stratum", "selected", "random_key", "rank_in_stratum", "skill_id", "repo", "skill_path", "file_sha",
                     "skill_md_sha256", "copies_in_dataset", "has_scripts", "n_script_files", "has_scripts_gitskills",
                     "n_program_source_files", "n_files_listed", "sibling_count", "n_files_without_text",
                     "listing_captured", "listing_complete", "folder_complete", "repo_stars", "repo_language"]
    frame.sort_values(["stratum", "file_sha"])[frame_columns].to_csv(OUT_DIR / "sampling_frame.csv", index=False, lineterminator="\n",
                                                                     float_format="%.17g")
    created_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    metadata = {
        "sample_id": SAMPLE_ID, "created_at": created_at, "random_seed": SEED, "sampling_method": METHOD,
        "random_number_generator": f"Python {platform.python_version()} random.Random({SEED}).random(); one generator per "
                                   "stratum; Python guarantees this sequence for an integer seed across versions",
        "database": {"path": str(DB_PATH.relative_to(ROOT)) if DB_PATH.is_relative_to(ROOT) else str(DB_PATH),
                     "sha256": DB_SHA256, "bytes": DB_PATH.stat().st_size, "identified_as": DATASET_LABEL},
        "population": population,
        "unit_of_sampling": "representative SKILL.md (artifacts.dedup_primary = 1), one per distinct content hash (file_sha)",
        "inclusion_criteria": [
            "representative row (dedup_primary = 1) with non-empty SKILL.md text",
            "folder listing captured (composition_fetched = 1)",
            "assigned to a stratum by the ordered rules in sampling_report.md",
            "no selection by keyword, repository popularity, earlier pilots or security signals",
        ],
        "exclusions": exclusions,
        "strata": {s: {"description": STRATUM_DESCRIPTIONS[s], "population_N": int(strata.loc[s, "population_N"]),
                       "sample_n": int(strata.loc[s, "sample_n"]), "sampling_fraction": float(strata.loc[s, "sampling_fraction"]),
                       "design_weight": float(strata.loc[s, "design_weight"]),
                       "complete_folders_in_population": int(strata.loc[s, "complete_folders_in_population"]),
                       "complete_folders_in_sample": int(manifest.loc[manifest["stratum"] == s, "folder_complete"].sum())}
                   for s in STRATUM_SIZES},
        "script_extensions": sorted(SCRIPT_EXTENSIONS), "program_source_extensions": sorted(PROGRAM_SOURCE_EXTENSIONS),
        "weighting_note": "Equal allocation: population-level estimates must weight each skill by design_weight = N_h / n_h.",
        "overlap_with_previous_selections": overlap,
    }
    (OUT_DIR / "sampling_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    (OUT_DIR / "sample_manifest.json").write_text(json.dumps({"metadata": metadata, "skills": json.loads(manifest.to_json(orient="records"))},
                                                             indent=2) + "\n", encoding="utf-8")
    md_rows = lambda df: "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False))
    report = f"""# Sample `{SAMPLE_ID}`

Created {created_at} from `{metadata['database']['path']}` (SHA-256 `{DB_SHA256[:16]}...`).
**Dataset:** {DATASET_LABEL}.

**Method:** {METHOD}. Seed {SEED}.

**Population:** {population['skill_md_occurrences']:,} SKILL.md occurrences, {population['distinct_content_hashes']:,} distinct contents,
{population['representatives']:,} representatives (one per content hash). Eligible after exclusions: {int(strata['population_N'].sum()):,}.

## Exclusions

| reason | representatives |
|---|---|
{md_rows(pd.DataFrame(list(exclusions.items())))}

## Strata

| stratum | definition | population N | sample n | sampling fraction | design weight | complete folders in sample |
|---|---|---|---|---|---|---|
{md_rows(pd.DataFrame([(s, STRATUM_DESCRIPTIONS[s], int(strata.loc[s,'population_N']), STRATUM_SIZES[s], f"{strata.loc[s,'sampling_fraction']:.4f}",
                         f"{strata.loc[s,'design_weight']:.1f}", f"{metadata['strata'][s]['complete_folders_in_sample']} of {STRATUM_SIZES[s]}") for s in STRATUM_SIZES]))}

Strata are assigned by ordered rules: (1) no listing -> excluded; (2) any executable script -> scripts; (3) truncated
listing -> excluded; (4) program source without scripts -> excluded; (5) any supporting file -> supporting_no_scripts;
(6) otherwise -> skill_md_only. Executable scripts: extensions {', '.join(sorted(SCRIPT_EXTENSIONS))}, or an extensionless file starting with `#!`.

**Weighting.** Equal allocation over strata of very different sizes: within-stratum results can be reported
directly; population-level proportions must weight each skill by `design_weight`.

**Overlap with earlier pilots / keyword selections:** {len(overlap)} skill(s){' — ' + '; '.join(o['previous_selection'] for o in overlap) if overlap else ''}.

## Selected skills

| # | stratum | repository | path | files listed | scripts | folder complete |
|---|---|---|---|---|---|---|
{md_rows(manifest[['sample_position', 'stratum', 'repo', 'skill_path', 'n_files_listed', 'n_script_files', 'folder_complete']])}
"""
    (OUT_DIR / "sampling_report.md").write_text(report, encoding="utf-8")
    print(f"Saved to {show(OUT_DIR)}:")
    for path in sorted(OUT_DIR.iterdir()):
        print(f"  {path.name:24} sha256 {sha256_text(path)[:16]}...")